# Time-dependent delivery routing for ADEME — Modeling deliverable

**CesiCDP project team** · Response to the ADEME call for expressions of interest on new mobility solutions

CesiCDP wants to show ADEME that Operations Research can cut the time, energy and emissions of delivery rounds. This notebook is the first step: before writing any algorithm, we define precisely **what problem we solve** and **how hard it is**.

The problem, called **TD-CVRP-TW** in the rest of the notebook, is to plan the routes of $k$ identical trucks of capacity $Q$ that leave a depot, serve every customer once and come back, so that the **total duration** of the routes is minimal. Travel times depend on the **time slot** (expected traffic), and in the advanced version every customer has a **time window** for the delivery, with waiting allowed if the truck arrives early.

**Storyline of the notebook**

| Section | Question answered |
|---|---|
| 1. Context and problem statement | Why does ADEME care, and what exactly do we want to optimize? |
| 2. Formalization of data and assumptions | Which data do we need, and what do we assume about it? |
| 3. Integer Linear Programming model | How can the problem be written as an exact mathematical model? |
| 4. Complexity | How hard is the problem, and what does this imply for solving large instances? |
| 5. Conclusion and next steps | What have we established, and what comes next? |
| Appendix A – Notation | Every symbol used, with its Python name |

## 1. Context and problem statement

### 1.1 Context
ADEME has issued a call for expressions of interest to support demonstrators and pilot projects for new mobility solutions for people and goods, adapted to different types of territories. CesiCDP is already active in *Smart Multimodal Mobility*, having led several studies with partners. New transport technologies (electric, low-emission vehicles) are cheaper to run and cleaner, but they raise new challenges, especially in **optimizing resource management**.

Transport logistics is therefore a major issue for the future, with many applications: mail delivery, parcel delivery, road-network management, waste collection. The environmental impact of better routing (less time on the road, less congestion exposure, less fuel or energy) is potentially large.

Our team (4 people) answers the call to secure new contracts with attractive funding. CesiCDP chose to focus on **delivery route management**.

### 1.2 Problem statement
A depot, a road network and a set of customers are given. Customers must each be visited once by one of $k$ identical trucks of capacity $Q$. Each truck leaves the depot, serves a subset of customers and returns to the depot. The time needed to traverse a road segment depends on the **time slot** (expected traffic). We must choose which customers go in which route, the order of visits and the departure times so as to **minimize the total duration of the routes**.

The method must come from **Operations Research** and scale to **several thousand customers**.

### 1.3 Scope
| Level | Content |
|---|---|
| Basic version | model, Python code for large instances, statistical study of experimental behavior, fleet of $k$ identical trucks |
| Added constraints (to be more realistic and attractive for ADEME) | customer time windows $[a_i,b_i]$; service times $s_i$ |

The added constraints make the problem more difficult, which is accepted: it shows the realism and ambition of the proposal. The resulting problem is called **TD-CVRP-TW**.

### 1.4 Why it matters (arguments for ADEME)
- Real travel times vary strongly over the day; ignoring this gives routes that are optimistic and late in practice.
- Shorter total duration means lower energy use and emissions, and a better use of the fleet and drivers.
- The same model covers several applications (delivery, mail, waste collection) by changing demands and windows.
- The formal model supports later extensions: emissions-aware objective, electric fleet, uncertain traffic.

### 1.5 Deliverable expected at this stage
Modeling check (Prosit 3, not assessed): context, formal data/problem/objective, link with classical problems and complexity proof, first ILP (variables, objective, constraints), bibliographic references. **No solution method is required yet.**

## 2. Formalization of data and assumptions

### 2.1 Input data
| Data | Symbol | Type / unit | Possible source |
|---|---|---|---|
| Road network | $G=(V,A)$ | directed graph | OpenStreetMap |
| Arc length | $d_a>0$ | km | OpenStreetMap geometry |
| Time slots | $0=\beta_0<\dots<\beta_K=T$ | minutes | chosen by the team (e.g. 5 slots over 12 h) |
| Speed per slot | $v_a^{s}>0$ | km/min | historical or predicted traffic data (open data, floating-car data) or a synthetic profile |
| Depot | node $0$ | node of $V$ | client |
| Customers | $C$, $n=\lvert C\rvert$ | subset of $V\setminus\{0\}$ | client orders |
| Demand | $q_i>0$ | units | orders |
| Service time | $s_i\ge0$ | minutes | estimated per stop |
| Time window | $[a_i,b_i]$ | minutes | client delivery slots |
| Fleet | $k$, $Q$ | integer, units | client fleet |

For the experiments, instances are **generated** (random points, synthetic congestion profiles with morning and evening peaks) and, if possible, completed with real road data.

### 2.2 Derived data
- **Travel time** $\tau_a(t)$ from the speeds, with speed changes when a breakpoint is crossed (Ichoua et al., 2003). Beyond $T$ the last slot's speed applies.
- **Customer graph** $H=(N,E)$, $N=C\cup\{0\}$, with $\tau_{ij}(t)$ obtained by time-dependent shortest paths on $G$.
- **Slot travel times** for the ILP: $\tau^{s}_{ij}$, the travel time from $i$ to $j$ when leaving during slot $s$, computed with the slot-$s$ speeds (see Appendix A.6). The ILP uses this simpler *departure-slot* rule, which keeps the model linear; the exact Ichoua model above will be used by the heuristic and to evaluate its routes.

### 2.3 Assumptions
| # | Assumption | Justification / consequence |
|---|---|---|
| A1 | One depot; all trucks start and end there | standard in delivery; simplifies the model |
| A2 | All trucks are identical (capacity $Q$) and at most $k$ are used | basic version required by the project |
| A3 | Each customer is visited exactly once by one truck (no split delivery) | realistic for parcels; keeps the model simple |
| A4 | Demands are deliveries from the depot; load decreases along a route; $q_i\le Q$ | otherwise the instance is trivially infeasible |
| A5 | Travel times are deterministic and known (predicted traffic per slot) | stochastic traffic is a later extension |
| A6 | Speeds are piecewise constant per slot and the travel time follows the model of Ichoua et al., so the **FIFO** property holds | avoids unrealistic "leave later, arrive earlier" |
| A7 | Trucks may pass through other nodes without serving them | the customer graph uses shortest paths |
| A8 | A truck may wait at a customer if it arrives before the window opens; service must **start** within $[a_i,b_i]$ | usual convention for time windows |
| A9 | All trucks leave the depot at time $0$ | the duration of a route is its return time, which keeps the ILP simple; a free departure time (not counting waiting at the depot) is a planned refinement |
| A10 | Objective: sum over trucks of (return time $-$ departure time) | matches "minimize the total duration of the route" |
| A11 | All routes must be completed before $T$ | finite horizon |
| A12 | In the ILP, the travel time of a trip is fixed by the slot in which the truck leaves ($\tau^{s}_{ij}$) | keeps the ILP linear; it approximates the exact Ichoua travel time, which the heuristic will use |
| A13 | A route has no maximum duration and drivers have no break constraints | possible future extension |
| A14 | Single speed profile for all vehicle types (no emission model yet) | possible future extension |

### 2.4 Data validity conditions (to check on every instance)
1. $d_a>0$ and $v_a^s>0$ for all $a,s$.
2. $0<q_i\le Q$ for all $i$; for feasibility with $k$ trucks, $\sum_i q_i\le kQ$.
3. $0\le a_i\le b_i\le T$.
4. $G$ strongly connected on the nodes used (every customer reachable from and back to the depot).
5. Window feasibility (necessary condition): each customer must be reachable in time, i.e. $\alpha_{0i}(0)\le b_i$, and the depot must be reachable again before $T$ after serving it, i.e. $\max(a_i,\alpha_{0i}(0))+s_i+\tau_{i0}(\cdot)\le T$.

### 2.5 Open questions to settle with the team and tutor
- Final choice of $K$ and $T$.
- Free departure time from the depot (A9) and exact Ichoua travel times inside the ILP (A12): worth the extra model complexity for the final deliverable?
- Real traffic data or synthetic profiles for the experiments?
- Keep the objective as total duration only, or add a vehicle fixed cost?
- Confirm that both added constraints (time windows and service times) are kept in the final scope.

## 3. Integer Linear Programming model

With the data and assumptions fixed, we can write the problem as an Integer Linear Program. All symbols follow the shared notation sheet (Appendix A, in particular A.6).

### 3.1 Sets and parameters

| Symbol | Meaning |
|---|---|
| $C=\{1,\dots,n\}$ | customers |
| $0$ | depot, as the **start** of a route |
| $n+1$ | copy of the depot, as the **end** of a route |
| $N=C\cup\{0\}$ | possible origins of an arc |
| $N^{+}=C\cup\{n+1\}$ | possible destinations of an arc |
| $\{1,\dots,K\}$, $s$ | time slots, slot $s=[\beta_{s-1},\beta_s)$, with $\beta_0=0$ and $\beta_K=T$ |
| $\tau^{s}_{ij}$ | travel time from $i$ to $j$ when **leaving $i$ during slot $s$**: shortest-path time in $G$ with the slot-$s$ speeds, $\tau^{s}_{ij}=\min_{P:\,i\to j}\sum_{a\in P} d_a/v^{s}_a$; $\tau^{s}_{i,n+1}=\tau^{s}_{i0}$ |
| $q_i$, $s_i$, $[a_i,b_i]$ | demand, service time, time window of customer $i$ |
| $k$, $Q$ | number of trucks, capacity of each truck |
| $T$ | planning horizon: every truck must be back at the depot by $T$ |
| $\varepsilon>0$ | time resolution (e.g. 1 second), used to make slots half-open |
| $M_{ij}=T+\max_s\tau^{s}_{ij}$, $M_i=T+\max_s\tau^{s}_{i0}$ | big-M constants, large enough to deactivate (C8) and (C10) |

**Modelling choices for this version** (assumptions A9 and A12 in Section 2.3):
- Every truck leaves the depot at time $0$. The route duration is then the return time, so the objective is the sum of return times.
- Travel time is fixed by the slot in which the truck **leaves** $i$ (departure-slot approximation). The exact travel-time model of Ichoua et al. (2003), where the speed changes when a slot boundary is crossed, will be used by the heuristic.

### 3.2 Decision variables

| Variable | Domain | Meaning |
|---|---|---|
| $x_{ij}$, $i\in N$, $j\in N^{+}$, $i\ne j$ | $\{0,1\}$ | 1 if some truck drives directly from $i$ to $j$ (the arc $(0,n+1)$ is not defined) |
| $z^{s}_{i}$, $i\in C$, $s\in\{1,\dots,K\}$ | $\{0,1\}$ | 1 if the truck **leaves** $i$ during slot $s$ |
| $r_i$ | $\ge 0$ | arrival time at customer $i$ |
| $u_i$ | $[a_i,b_i]$ | start of service at $i$ |
| $t_i$ | $[0,T)$ | departure time from $i$ |
| $L_i$ | $[q_i,Q]$ | cumulative load delivered on the route up to and including $i$ |
| $f_i$ | $[0,T]$ | return time to the depot **if $i$ is the last customer of its route**, 0 otherwise |

Waiting time at $i$ is $w_i=u_i-r_i\ge0$.

### 3.3 Objective

$$
\min \sum_{i\in C} f_i \qquad (O)
$$

Only the last customer of each route has $f_i>0$, so (O) is the sum of the return times. Since every route leaves the depot at time 0, this is the total duration of the routes (A10).

### 3.4 Constraints
**Routing structure (basic)**

$$
\sum_{i\in N\setminus\{j\}}x_{ij}=1\quad\forall j\in C \qquad (C1)
$$

$$
\sum_{h\in N^{+}\setminus\{j\}}x_{jh}=1\quad\forall j\in C \qquad (C2)
$$

$$
\sum_{j\in C}x_{0j}\le k \qquad (C3)
$$

(C1)–(C2): every customer has exactly one predecessor and one successor, i.e. is visited once. (C3): at most $k$ routes leave the depot. Flow conservation at the depot is implied: the number of arcs entering $n+1$ equals the number leaving $0$.

**Capacity and subtour elimination (basic)**

$$
L_j\ \ge\ L_i+q_j-Q\,(1-x_{ij})\quad\forall i,j\in C,\ i\ne j,\qquad q_j\le L_j\le Q \qquad (C4)
$$

If $j$ follows $i$, the load grows by $q_j$; otherwise the constraint is slack (because $L_i\le Q$). Since $L_j\ge q_j$, the first customer of a route needs no special case.

(C4) is the capacity version of the Miller–Tucker–Zemlin constraints (Miller et al., 1960; Kulkarni & Bhave, 1985) and it also **eliminates subtours**. Suppose $x$ contains a cycle $i_1\to i_2\to\dots\to i_m\to i_1$ made only of customers. Adding (C4) along the cycle gives $0\ge\sum_{p=1}^{m}q_{i_p}>0$, because every $q_i>0$ (A4), which is impossible. So every chain of customers must start at $0$ and end at $n+1$. The time constraints (C8) eliminate subtours in the same way whenever travel times are positive.

**Time-dependent travel (basic)**

$$
\sum_{s=1}^{K}z^{s}_{i}=1\quad\forall i\in C \qquad (C5a)
$$

$$
\sum_{s=1}^{K}\beta_{s-1}z^{s}_{i}\ \le\ t_i\ \le\ \sum_{s=1}^{K}(\beta_s-\varepsilon)\,z^{s}_{i}\quad\forall i\in C \qquad (C5b)
$$

$$
t_i\ \ge\ u_i+s_i\quad\forall i\in C \qquad (C6)
$$

$$
r_j\ \ge\ \tau^{1}_{0j}\,x_{0j}\quad\forall j\in C \qquad (C7)
$$

$$
r_j\ \ge\ t_i+\sum_{s=1}^{K}\tau^{s}_{ij}\,z^{s}_{i}-M_{ij}(1-x_{ij})\quad\forall i,j\in C,\ i\ne j \qquad (C8)
$$

(C5) selects the slot of departure; $\varepsilon$ makes slots half-open and removes the ambiguity at slot boundaries. (C6): a truck leaves after finishing the service. (C7): the truck leaves the depot at time 0, hence in slot 1. (C8): arrival at $j$ is at least departure time plus the travel time of the slot of departure. The product $\tau^{s}_{ij}\,z^{s}_{i}$ is **linear** because $\tau^{s}_{ij}$ is a constant. Because $t_i$ (departure) and not the arrival is used, waiting before leaving is allowed, so the model stays valid even if the departure-slot approximation breaks FIFO at a slot boundary.

**Time windows and waiting (advanced)**

$$
u_i\ \ge\ r_i,\qquad a_i\le u_i\le b_i\quad\forall i\in C \qquad (C9)
$$

$u_i\ge r_i$ and $u_i\ge a_i$ let the truck **wait** until the window opens; $u_i\le b_i$ **forbids** serving after the window closes. For the basic version, set $a_i=0$ and $b_i=T$.

**Return to the depot and horizon**

$$
f_i\ \ge\ t_i+\sum_{s=1}^{K}\tau^{s}_{i0}\,z^{s}_{i}-M_i\,(1-x_{i,n+1}),\qquad f_i\le T\,x_{i,n+1}\quad\forall i\in C \qquad (C10)
$$

If $i$ is the last customer, $f_i$ is at least the return time and at most $T$ (A11); otherwise $f_i=0$.

### 3.5 Full model (summary)

$$
\begin{aligned}
(O)\quad & \min\ \textstyle\sum_{i\in C} f_i\\
(C1)\quad & \textstyle\sum_{i\in N\setminus\{j\}}x_{ij}=1 \quad \forall j\in C\\
(C2)\quad & \textstyle\sum_{h\in N^{+}\setminus\{j\}}x_{jh}=1 \quad \forall j\in C\\
(C3)\quad & \textstyle\sum_{j\in C}x_{0j}\le k\\
(C4)\quad & L_j\ge L_i+q_j-Q(1-x_{ij}) \quad \forall i,j\in C,\ i\ne j\\
(C5a)\quad & \textstyle\sum_{s}z^{s}_{i}=1 \quad \forall i\in C\\
(C5b)\quad & \textstyle\sum_{s}\beta_{s-1}z^{s}_{i}\le t_i\le\sum_{s}(\beta_s-\varepsilon)z^{s}_{i} \quad \forall i\in C\\
(C6)\quad & t_i\ge u_i+s_i \quad \forall i\in C\\
(C7)\quad & r_j\ge\tau^{1}_{0j}x_{0j} \quad \forall j\in C\\
(C8)\quad & r_j\ge t_i+\textstyle\sum_{s}\tau^{s}_{ij}z^{s}_{i}-M_{ij}(1-x_{ij}) \quad \forall i,j\in C,\ i\ne j\\
(C9)\quad & u_i\ge r_i,\ \ a_i\le u_i\le b_i \quad \forall i\in C\\
(C10)\quad & f_i\ge t_i+\textstyle\sum_{s}\tau^{s}_{i0}z^{s}_{i}-M_i(1-x_{i,n+1}),\ \ f_i\le T\,x_{i,n+1} \quad \forall i\in C\\
& x_{ij}\in\{0,1\},\ z^{s}_{i}\in\{0,1\},\ r_i\ge0,\ t_i\in[0,T),\ q_i\le L_i\le Q,\ f_i\ge0
\end{aligned}
$$

**Size of the model.** $n(n+1)$ binary variables $x_{ij}$, $nK$ binary variables $z^{s}_{i}$, $5n$ continuous variables, and $2n(n-1)+O(n)$ constraints, dominated by (C4) and (C8). For $n=1000$ customers this is already about two million constraints with weak big-M relaxations, which is why the ILP is only expected to solve small instances exactly (see Section 4).

## 4. Complexity

The ILP gives an exact model, but how hard is it to solve? This section places the problem in the literature and proves that it is NP-hard, which tells us what the ILP can and cannot be used for.

### 4.1 Position of the problem in the literature

Our problem, **TD-CVRP-TW**, combines four features that each have a long history in Operations Research. Each line of the table adds one feature to the line above, so every problem in the table is a **special case** of the ones below it.

| Problem | Feature added | Reference |
|---|---|---|
| TSP | one vehicle, visit every city once, return to the start, minimize the length | Karp (1972), Garey & Johnson (1979) |
| CVRP | $k$ vehicles of capacity $Q$, each customer has a demand $q_i$ | Dantzig & Ramser (1959), Toth & Vigo (2014) |
| VRPTW | service times $s_i$ and time windows $[a_i,b_i]$, waiting allowed | Solomon (1987) |
| TDVRP | travel times depend on the departure time (traffic per slot) | Malandraki & Daskin (1992), Ichoua et al. (2003) |
| **TD-CVRP-TW** (ours) | all of the above, objective = total duration of the routes | Dabia et al. (2013), Gendreau et al. (2015) |

Two remarks place our problem more precisely:
- With time-dependent travel times, **minimizing duration is not the same as minimizing distance**. A shorter road taken at rush hour can be slower than a longer one, and a later departure can avoid a traffic peak. This is why we minimize duration (A10) rather than distance.
- The closest problem in the literature is the time-dependent VRP with time windows where the objective is the total route duration, studied by Dabia et al. (2013) with the Ichoua et al. (2003) travel-time model that we also use (A6).

The rest of this section proves that the decision version of TD-CVRP-TW is **NP-complete**, so the optimization problem is **NP-hard**. The proof uses the oldest problem of the table, the TSP.

### 4.2 NP-hardness: reduction from the TSP

#### 4.2.1 Decision version of our problem

Complexity classes are defined for yes/no questions, so we first state the decision version of the problem.

> **TD-CVRP-TW-D**
> *Instance:* road network $G=(V,A)$ with lengths $d_a$, time slots $\beta_0<\dots<\beta_K$ and speeds $v^{s}_a$; depot $0$, customers $C$ with demands $q_i$, service times $s_i$ and windows $[a_i,b_i]$; fleet size $k$, capacity $Q$, horizon $T$; a bound $B$.
> *Question:* is there a set of at most $k$ routes, each leaving the depot at time $0$ (A9), such that every customer is served exactly once, the load of every route is at most $Q$, every service starts inside its window, every truck is back by $T$, and the **total duration is at most $B$**?

If we could solve the optimization problem in polynomial time, we could answer TD-CVRP-TW-D in polynomial time by comparing the optimum with $B$. So proving that TD-CVRP-TW-D is NP-hard proves that the optimization problem is NP-hard too.

#### 4.2.2 Source problem: metric TSP

> **$\Delta$-TSP-D**
> *Instance:* cities $\{0,\dots,m-1\}$, integer costs $c_{ij}=c_{ji}\ge1$ satisfying the triangle inequality $c_{ij}\le c_{ih}+c_{hj}$, a bound $B$.
> *Question:* is there a tour (a cycle visiting every city exactly once) of cost at most $B$?

$\Delta$-TSP-D is NP-complete (Garey & Johnson, 1979, problem ND22). The classical proof reduces Hamiltonian Cycle (NP-complete, Karp 1972) to it: give cost $1$ to the edges of the graph and cost $2$ to the non-edges, and ask for a tour of cost at most $m$. Costs in $\{1,2\}$ always satisfy the triangle inequality, since $c_{ij}\le2\le c_{ih}+c_{hj}$. The TSP is therefore NP-complete **even with the triangle inequality and costs in $\{1,2\}$** (Papadimitriou & Yannakakis, 1993).

**Why we need the triangle inequality.** In our problem a truck may drive through a node without serving it (A7), so the travel time between two customers is a **shortest-path** time. Shortest-path times always satisfy the triangle inequality. With metric costs, the direct road is already a shortest path, so the travel times in our instance are exactly the TSP costs.

#### 4.2.3 The reduction

Given an instance $(m,c,B)$ of $\Delta$-TSP-D, we build the following instance of TD-CVRP-TW-D:

| Element of our instance | Value | Effect |
|---|---|---|
| Road network $G$ | $V=\{0,\dots,m-1\}$, an arc in each direction between every pair of cities, $d_{ij}=c_{ij}$ | one node per city |
| Time slots and speeds | $K=1$, $v^{1}_a=1$ km/min on every arc | $\tau_a(t)=d_a$ for all $t$: no traffic effect, FIFO holds |
| Depot and customers | depot $=$ city $0$, $C=\{1,\dots,m-1\}$, so $n=m-1$ | every city must be visited |
| Fleet | $k=1$, $q_i=1$, $Q=n$ | one truck that can carry everything |
| Service and windows | $s_i=0$, $a_i=0$, $b_i=T$ | windows never bind |
| Horizon | $T=m\cdot c_{\max}$, where $c_{\max}=\max c_{ij}$ | larger than the cost of any tour |
| Bound | $B$ (unchanged) | |

**Polynomial time.** The instance has $m$ nodes and $m(m-1)$ arcs, and its numbers are the costs $c_{ij}$, the constants $0$, $1$, $n$, and $T$, whose binary size is $O(\log m+\log c_{\max})$. The construction takes $O(m^2)$ time.

**Travel times.** With one slot and speed 1, $\tau_a(t)=d_a$ for every $t$. By the triangle inequality, the direct arc $(i,j)$ is a shortest path from $i$ to $j$, so $\tau_{ij}(t)=c_{ij}$ for every departure time $t$.

**What a solution looks like.** With $k=1$, a feasible solution is a single route $R=(0,\pi_1,\dots,\pi_n,0)$ where $(\pi_1,\dots,\pi_n)$ is an ordering of all the customers. The route leaves at time $0$, there is no service time, and waiting can only add time, so its duration is

$$
D(R)=c_{0\pi_1}+c_{\pi_1\pi_2}+\dots+c_{\pi_{n-1}\pi_n}+c_{\pi_n0}+(\text{total waiting})\ \ge\ c(0,\pi_1,\dots,\pi_n,0),
$$

with equality when the truck never waits, which is always allowed since $a_i=0$.

#### 4.2.4 Correctness

**($\Rightarrow$) If the TSP instance is a yes-instance, so is ours.** Take a tour of cost at most $B$ and rotate it so that it starts at city $0$: $(0,\pi_1,\dots,\pi_n,0)$. Use it as the single route, with no waiting. Every customer is served once. The load is $n=Q$. Every arrival time is at most the tour cost, which is at most $m\cdot c_{\max}=T=b_i$, so all windows and the horizon are respected. The duration equals the tour cost, so it is at most $B$.

**($\Leftarrow$) If our instance is a yes-instance, so is the TSP instance.** Take a feasible solution of duration at most $B$. Because $k=1$, it is one route $(0,\pi_1,\dots,\pi_n,0)$ visiting every customer exactly once, so $(0,\pi_1,\dots,\pi_n)$ is a tour of all $m$ cities. By the inequality above, its cost is at most $D(R)\le B$.

So $\Delta\text{-TSP-D}\le_p\text{TD-CVRP-TW-D}$, and **TD-CVRP-TW-D is NP-hard**.

#### 4.2.5 What the reduction tells us

- **The basic version is already NP-hard.** The instance built above has no binding time windows, no service times and a single slot, so TD-CVRP (the basic version) is NP-hard. The advanced constraints cannot make the problem easier, because the basic version is a special case of the advanced one.
- **The problem is strongly NP-hard.** Starting from TSP instances with costs in $\{1,2\}$, every number in our instance is at most $2m$. Hence the problem stays NP-hard even when all numbers are small, and no pseudo-polynomial algorithm exists unless P = NP (Garey & Johnson, 1979).
- **The result does not depend on our modelling choices.** The reduction uses one slot and constant speeds, so it holds whether the departure time from the depot is fixed or free (A9), and with either the departure-slot rule of the ILP or the exact Ichoua model (A12).

#### 4.2.6 Even finding a feasible solution is NP-hard

For the CVRP alone, a feasible solution is easy to find when trucks are unlimited: one truck per customer. In our problem, the fleet size $k$ and the horizon $T$ are fixed, and then even **feasibility** is NP-complete.

- **Fleet size and capacity (Bin Packing).** Bin Packing asks whether items of sizes $w_1,\dots,w_n$ fit into $k$ bins of capacity $Q$, and it is strongly NP-complete (Garey & Johnson, 1979, problem SR1). Build an instance with one customer per item ($q_i=w_i$), the same $k$ and $Q$, all travel times equal to 1, $s_i=0$, $a_i=0$, $b_i=T$ and $T=n+1$. Any route of at most $n$ customers takes at most $n+1=T$ minutes, so time never binds. A feasible set of routes exists exactly when the items fit into $k$ bins.
- **Horizon and time windows.** In the reduction of Section 4.2.3, replace the horizon by $T=B$. The single route is then feasible exactly when the tour costs at most $B$, so deciding feasibility is as hard as the TSP. Savelsbergh (1985) proved the same result for the TSP with time windows: finding a feasible solution is strongly NP-complete.

As a consequence, unless P = NP, no polynomial-time algorithm can guarantee to return a feasible solution whenever one exists, and so no polynomial-time algorithm has a guaranteed approximation ratio. The solution method will have to deal with routes that break capacity or windows during the search.

### 4.3 NP membership and conclusion

#### 4.3.1 TD-CVRP-TW-D is in NP

A problem is in NP if every yes-instance has a certificate of polynomial size that can be checked in polynomial time.

**Certificate.** The list of at most $k$ routes, each given as the ordered list of its customers. Its size is $O(n\log n)$ bits.

**Verification.** For each route $R_\ell=(0,\pi_1,\dots,\pi_{m_\ell},0)$:
1. Check that every customer of $C$ appears in exactly one route and that there are at most $k$ routes: $O(n)$.
2. Check that the total demand of each route is at most $Q$: $O(n)$.
3. Compute the **earliest schedule**: leave the depot at $t=0$; at each customer $j$ reached from $i$, set $r_j=\alpha_{ij}(t_i)$, $u_j=\max(r_j,a_j)$, check $u_j\le b_j$, then set $t_j=u_j+s_j$. At the end, check that the return time $\alpha_{\pi_{m_\ell}0}(t_{\pi_{m_\ell}})$ is at most $T$.
4. Check that the sum of the return times is at most $B$.

**Why the earliest schedule is enough.** Under FIFO (A6), $\alpha_{ij}(t)$ is non-decreasing in $t$, and so are $\max(\cdot,a_j)$ and $+s_j$. Their composition is non-decreasing, so leaving as early as possible never makes a later arrival, service start or return time larger. If some schedule of these routes respects all windows and has total duration at most $B$, the earliest schedule does too.

**Why each step is polynomial.**
- *Travel times.* $\alpha_{ij}(t)$ is an earliest-arrival time in $G$. Under FIFO, it is computed by Dijkstra's algorithm adapted to time-dependent arcs (Dreyfus, 1969; Kaufman & Smith, 1993), which evaluates each arc once and runs in $O(\lvert A\rvert(K+\log\lvert V\rvert))$ time: one arc evaluation $\tau_a(t)$ in the Ichoua model walks through at most $K$ slots, so it costs $O(K)$.
- *Numbers.* All computed times are rationals obtained from the input by polynomially many additions, multiplications and divisions, so their binary size stays polynomial.

Step 3 makes $n+k$ such computations, so the whole check runs in $O\big((n+k)\,\lvert A\rvert(K+\log\lvert V\rvert)\big)$ time and **TD-CVRP-TW-D is in NP**.

**The ILP's travel-time rule.** With the departure-slot rule of the ILP (A12), leaving later can occasionally arrive earlier at a slot boundary, so FIFO may fail. The certificate then also gives the departure slot of each customer (the values $z^{s}_i$ of the ILP), and the verifier leaves at $t_i=\max(u_i+s_i,\beta_{s-1})$ and checks $t_i<\beta_s$. Inside a slot the travel time is constant, so leaving later in the same slot never helps. The check stays polynomial.

*Note:* membership for a **free** depot departure time (the planned refinement of A9) would need the departure times in the certificate, together with a proof that optimal departure times have polynomial size. We leave this for the final deliverable.

#### 4.3.2 Conclusion

| Version | Complexity | Proof |
|---|---|---|
| TD-CVRP-TW-D (decision, advanced) | **NP-complete**, strongly | 4.2 (NP-hard) + 4.3.1 (in NP) |
| TD-CVRP-D (decision, basic) | **NP-complete**, strongly | same reduction, no windows used |
| TD-CVRP-TW and TD-CVRP (optimization) | **NP-hard** | an optimal solution answers the decision version |
| Finding any feasible solution ($k$ and $T$ fixed) | **NP-complete** | 4.2.6 (Bin Packing; TSP with $T=B$) |

**Consequences for the project.**
- Unless P = NP, no algorithm solves every instance optimally in polynomial time. The solve time of the ILP grows exponentially in the worst case, and even for the plain CVRP, state-of-the-art exact methods are limited to a few hundred customers (Toth & Vigo, 2014).
- Our target instances have several thousand customers. The ILP will therefore be used on small instances to obtain optimal solutions and bounds, and the large instances will need a heuristic or metaheuristic method (next deliverable).
- Since feasibility itself is hard when the fleet and the horizon are tight, the experiments should also measure how often the method finds a feasible solution, not only its cost.

## 5. Conclusion and next steps

**What this deliverable establishes**
- A precise statement of TD-CVRP-TW: data, assumptions A1–A14, and objective (total route duration).
- An ILP with capacity, MTZ-type subtour elimination, time-dependent travel times by slot, time windows and waiting (Section 3). It has $O(n^2)$ variables and constraints.
- A complexity proof: the decision version is **strongly NP-complete**, already in the basic version without time windows, and even finding a feasible solution is NP-complete when the fleet size and the horizon are fixed (Section 4).

**What this means for the next deliverables**
- The ILP will be used on small instances, to get optimal solutions and bounds against which the heuristic can be measured.
- Instances with several thousand customers will need a heuristic or metaheuristic method, which is the subject of the next deliverable.
- The experimental study will measure solution quality (gap to the best known solutions on VRPLIB instances), computation time as a function of $n$, and how often a feasible solution is found when windows are tight.

**Points we would like to discuss with the tutor**
- The two simplifications of the ILP: departure from the depot fixed at time $0$ (A9) and travel time fixed by the departure slot (A12).
- The open questions listed in Section 2.5.

## References

- Dabia, S., Ropke, S., van Woensel, T., & de Kok, T. (2013). Branch and price for the time-dependent vehicle routing problem with time windows. *Transportation Science*, 47(3), 380–396.
- Dantzig, G. B., & Ramser, J. H. (1959). The truck dispatching problem. *Management Science*, 6(1), 80–91.
- Dreyfus, S. E. (1969). An appraisal of some shortest-path algorithms. *Operations Research*, 17(3), 395–412.
- Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.
- Gendreau, M., Ghiani, G., & Guerriero, E. (2015). Time-dependent routing problems: A review. *Computers & Operations Research*, 64, 189–197.
- Ichoua, S., Gendreau, M., & Potvin, J.-Y. (2003). Vehicle dispatching with time-dependent travel times. *European Journal of Operational Research*, 144(2), 379–396.
- Karp, R. M. (1972). Reducibility among combinatorial problems. In R. E. Miller & J. W. Thatcher (Eds.), *Complexity of Computer Computations* (pp. 85–103). Plenum Press.
- Kaufman, D. E., & Smith, R. L. (1993). Fastest paths in time-dependent networks for intelligent vehicle-highway systems application. *IVHS Journal*, 1(1), 1–11.
- Kulkarni, R. V., & Bhave, P. R. (1985). Integer programming formulations of vehicle routing problems. *European Journal of Operational Research*, 20(1), 58–67.
- Malandraki, C., & Daskin, M. S. (1992). Time dependent vehicle routing problems: Formulations, properties and heuristic algorithms. *Transportation Science*, 26(3), 185–200.
- Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960). Integer programming formulation of traveling salesman problems. *Journal of the ACM*, 7(4), 326–329.
- Papadimitriou, C. H., & Yannakakis, M. (1993). The traveling salesman problem with distances one and two. *Mathematics of Operations Research*, 18(1), 1–11.
- Savelsbergh, M. W. P. (1985). Local search in routing problems with time windows. *Annals of Operations Research*, 4(1), 285–305.
- Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling problems with time window constraints. *Operations Research*, 35(2), 254–265.
- Toth, P., & Vigo, D. (Eds.). (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM.

## Appendix A – Notation
**Project:** Time-dependent delivery routing (CesiCDP, ADEME call) – **Problem name:** TD-CVRP-TW
*Single reference for all team members: every notebook section, the ILP and the code must use exactly these symbols.*

### A.1 Conventions
- Sets: capital letters ($V, A, C, N, N^{+}$). Parameters: lowercase/Latin ($q_i, s_i, d_a$). Decision variables: $x, z, r, u, t, L, f$.
- Node `0` is always the depot. Customers are `1..n`. In the ILP, node `n+1` is a copy of the depot used as the end of a route.
- Time unit: **minute**. Distance unit: **km**. Speed unit: **km/min**. Demand unit: **parcel units** (or kg, to be fixed per instance).
- A generic instant is written $t$ (minutes), as in $\tau_a(t)$. With a customer index, $t_i$ is the departure-time variable of the ILP.
- Python variable names are given in the last column of each table so code and maths stay aligned.

### A.2 Road network and traffic
| Symbol | Meaning | Python |
|---|---|---|
| $G=(V,A)$ | directed road network | `G` |
| $V$, $A$ | nodes (intersections, customer locations, depot) and arcs (road segments) | `V`, `A` |
| $a=(u,v)\in A$ | arc from $u$ to $v$ | `a` |
| $d_a>0$ | length of arc $a$ (km) | `length[a]` |
| $T$ | planning horizon end (min); horizon is $[0,T]$ | `T` |
| $K$ | number of time slots | `K` |
| $\beta_0=0<\beta_1<\dots<\beta_K=T$ | slot breakpoints (min) | `breakpoints` |
| $s\in\{1,\dots,K\}$ | slot index (slot $s$ = $[\beta_{s-1},\beta_s)$) | `s` |
| $v_a^{s}>0$ | speed on arc $a$ during slot $s$ (km/min) | `speed[a][s]` |
| $\tau_a(t)$ | travel time on arc $a$ when entering at time $t$ (min) | `tau_arc(a,t)` |
| $\alpha_a(t)=t+\tau_a(t)$ | arrival time on arc $a$ when entering at $t$ | `arrival_arc(a,t)` |
| FIFO | $t\mapsto\alpha_a(t)$ strictly increasing | – |

### A.3 Customers, depot and fleet
| Symbol | Meaning | Python |
|---|---|---|
| $0$ | depot | `0` |
| $C\subset V\setminus\{0\}$, $n=\lvert C\rvert$ | set and number of customers to serve | `C`, `n` |
| $N=C\cup\{0\}$ | decision nodes | `N` |
| $q_i>0$ ($q_0=0$) | demand of customer $i$ | `demand[i]` |
| $s_i\ge0$ ($s_0=0$) | service duration at $i$ (min) | `service[i]` |
| $[a_i,b_i]\subseteq[0,T]$ | time window for the **start** of service at $i$ | `tw[i]` |
| $k$ | maximum number of trucks | `k` |
| $Q$ | capacity of each truck (identical), $q_i\le Q$ | `Q` |

### A.4 Customer graph (reduced graph)
| Symbol | Meaning | Python |
|---|---|---|
| $H=(N,E)$ | complete directed graph on $N$ | `H` |
| $\tau_{ij}(t)$ | minimum travel time from $i$ to $j$ when leaving at $t$ (time-dependent shortest path in $G$) | `tau(i,j,t)` |
| $\alpha_{ij}(t)=t+\tau_{ij}(t)$ | earliest arrival at $j$ | `arrival(i,j,t)` |

### A.5 Solution (route-based notation)
| Symbol | Meaning |
|---|---|
| $r\le k$ | number of routes used |
| $R_\ell=(0,i^\ell_1,\dots,i^\ell_{m_\ell},0)$ | route $\ell$, with $m_\ell$ customers |
| $t_0^\ell$ | departure time of route $\ell$ from the depot (fixed to $0$ in the current ILP, see A9; a free departure time is a planned refinement) |
| $u_p^\ell$ | start of service of the $p$-th customer of route $\ell$ |
| $u^\ell_{m_\ell+1}$ | return time to the depot |
| $D(R_\ell)=u^\ell_{m_\ell+1}-t_0^\ell$ | duration of route $\ell$ |
| **Objective** | $\min\sum_{\ell=1}^r D(R_\ell)$ |

### A.6 ILP (slot-based formulation, Section 3)
| Symbol | Meaning | Python |
|---|---|---|
| $n+1$ | copy of the depot, end of a route | `n + 1` |
| $N^{+}=C\cup\{n+1\}$ | possible destinations of an arc (origins are $N$) | `N_plus` |
| $\tau^{s}_{ij}$ | travel time from $i$ to $j$ when leaving $i$ during slot $s$ (shortest path in $G$ with slot-$s$ speeds); $\tau^{s}_{i,n+1}=\tau^{s}_{i0}$ | `tau_slot[i][j][s]` |
| $\varepsilon$ | time resolution making slots half-open | `eps` |
| $M_{ij}$, $M_i$ | big-M constants, $T+\max_s\tau^{s}_{ij}$ and $T+\max_s\tau^{s}_{i0}$ | `M[i][j]`, `M_ret[i]` |
| $x_{ij}\in\{0,1\}$ | a truck drives directly from $i\in N$ to $j\in N^{+}$ | `x[i, j]` |
| $z^{s}_{i}\in\{0,1\}$ | the truck leaves $i$ during slot $s$ | `z[i, s]` |
| $r_i\ge0$ | arrival time at $i$ | `r[i]` |
| $u_i\in[a_i,b_i]$ | start of service at $i$ | `u[i]` |
| $t_i\in[0,T)$ | departure time from $i$ | `t[i]` |
| $w_i=u_i-r_i$ | waiting time at $i$ (derived, not a variable) | – |
| $L_i\in[q_i,Q]$ | cumulative load delivered up to and including $i$ | `L[i]` |
| $f_i\in[0,T]$ | return time to the depot if $i$ is the last customer of its route, else $0$ | `f[i]` |

> **Note on collisions:** $H=(N,E)$ is only the customer graph (the time-expanded formulation, which also used $H$ as a time index, was dropped). $A$ is the arc set of $G$ and $d_a$ an arc length, so the ILP uses $r_i$ for arrival and $t_i$ for departure. $s$ is the slot index *and* $s_i$ the service time; slots are always written as a superscript ($v_a^{s}$, $\tau^{s}_{ij}$, $z^{s}_{i}$), service times always carry a customer index as a subscript.

### A.7 Complexity vocabulary
| Symbol | Meaning |
|---|---|
| $\mathrm{P},\mathrm{NP}$ | classical complexity classes |
| $A\le_p B$ | polynomial-time reduction from $A$ to $B$ |
| HC | Hamiltonian Cycle |
| TSP, CVRP, VRPTW | travelling salesman, capacitated VRP, VRP with time windows |
| $\Delta$-TSP-D | decision TSP whose costs satisfy the triangle inequality |
| BP-D | decision Bin Packing |
| TD-CVRP | basic version (no time windows, no service times) |
| TD-CVRP-TW | full problem with time windows and service times |
| TD-CVRP-TW-D | decision version (is there a solution of duration $\le B$?) |
| $B$ | duration bound of the decision version |

### A.8 Instance sizes (for the statistical study later)
| Symbol | Meaning |
|---|---|
| $n$ | number of customers (up to several thousand) |
| $\lvert V\rvert,\lvert A\rvert$ | size of the road network |
| $k,Q$ | fleet size and capacity |
| $\rho=\sum_i q_i/(kQ)$ | fleet load ratio (tightness of capacity), proposed instance parameter |